
## Folder 09 — File 03: Incremental Delta MERGE:


- This notebook demonstrates how to perform incremental inserts and updates in Delta Lake using MERGE INTO.
- It covers all seven datasets, including the composite key for inventory. 
- It also checks for duplicate source keys before merging, because a Delta MERGE can fail or produce ambiguous results when multiple source rows match the same target row


                 E-COMMERCE LAKEHOUSE
                INCREMENTAL PROCESSING
                         |
                         v
        +--------------------------------------+
        | FILE 01: BRONZE INGESTION            |
        |                                      |
        | Auto Loader                          |
        | Checkpoints                          |
        | Schema Evolution                     |
        | Bronze Delta Tables                  |
        +--------------------------------------+
                         |
                         v
        +--------------------------------------+
        | FILE 02: SILVER PROCESSING           |
        |                                      |
        | Standardization                      |
        | Data Validation                      |
        | Deduplication                        |
        | Quarantine                           |
        +--------------------------------------+
                         |
                         v
        +--------------------------------------+
        | VALIDATED STAGING DELTA TABLE        |
        |                                      |
        | Only new, validated records          |
        | Waiting for MERGE                    |
        +--------------------------------------+
                         |
                         v
        +--------------------------------------+
        | FILE 03: INCREMENTAL DELTA MERGE     |
        |                                      |
        | Duplicate-Key Checks                 |
        | Seven MERGE Operations               |
        | Composite Business Key               |
        +--------------------------------------+
                         |
                         v
        +--------------------------------------+
        | SILVER DELTA TABLES                  |
        |                                      |
        | Insert New Records                   |
        | Update Existing Records              |
        +--------------------------------------+
                         |
                         v
        +--------------------------------------+
        | DELTA MERGE AUDIT REPORT             |
        |                                      |
        | Insert / Update Metrics              |
        | Execution Status                     |
        | Merge History                        |
        +--------------------------------------+


In [0]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    BooleanType,
    TimestampType
)

from datetime import datetime, timezone
import uuid


# ============================================================
# 2. ENVIRONMENT CONFIGURATION
# ============================================================

CATALOG = "workspace"

SILVER_SCHEMA = "ecommerce_silver"

AUDIT_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}."
    "incremental_delta_merge_audit_report"
)


# ============================================================
# 3. DATASET CONFIGURATION
# ============================================================

DATASETS = {
    "customers": "silver_customers",
    "products": "silver_products",
    "orders": "silver_orders",
    "order_items": "silver_order_items",
    "payments": "silver_payments",
    "shipments": "silver_shipments",
    "inventory_snapshots": "silver_inventory"
}


# ============================================================
# 4. EXECUTION CONFIGURATION
# ============================================================

RUN_ID = str(uuid.uuid4())

EXECUTION_TIMESTAMP = datetime.now(timezone.utc)

print("=" * 70)
print("INCREMENTAL DELTA MERGE AUDIT")
print("=" * 70)

print(f"Run ID: {RUN_ID}")
print(f"Execution Timestamp: {EXECUTION_TIMESTAMP}")
print(f"Total Datasets: {len(DATASETS)}")


# ============================================================
# 5. AUDIT RESULT SCHEMA
# ============================================================

audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("dataset_name", StringType(), False),
    StructField("target_table", StringType(), False),

    StructField("table_exists", BooleanType(), False),
    StructField("total_target_records", LongType(), True),

    StructField("merge_found", BooleanType(), False),
    StructField("merge_timestamp", TimestampType(), True),
    StructField("merge_operation", StringType(), True),

    StructField("inserted_rows", LongType(), True),
    StructField("updated_rows", LongType(), True),
    StructField("deleted_rows", LongType(), True),
    StructField("output_rows", LongType(), True),

    StructField("audit_status", StringType(), False),
    StructField("audit_details", StringType(), True),

    StructField("audit_timestamp", TimestampType(), False)
])


# ============================================================
# 6. HELPER FUNCTION
# ============================================================

def get_metric(metrics, *keys):

    if metrics is None:
        return None

    for key in keys:

        value = metrics.get(key)

        if value is not None:

            try:
                return int(value)
            except (ValueError, TypeError):
                return None

    return None


# ============================================================
# 7. DELTA MERGE AUDIT FUNCTION
# ============================================================

def audit_delta_merge(dataset_name, table_name):

    full_table_name = (
        f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    )

    print("\n" + "-" * 70)
    print(f"Auditing Dataset: {dataset_name}")
    print(f"Target Table: {full_table_name}")
    print("-" * 70)

    result = {
        "run_id": RUN_ID,
        "dataset_name": dataset_name,
        "target_table": full_table_name,

        "table_exists": False,
        "total_target_records": None,

        "merge_found": False,
        "merge_timestamp": None,
        "merge_operation": None,

        "inserted_rows": None,
        "updated_rows": None,
        "deleted_rows": None,
        "output_rows": None,

        "audit_status": "FAILED",
        "audit_details": None,

        "audit_timestamp": datetime.now(timezone.utc)
    }

    try:

        # ----------------------------------------------------
        # A. CHECK TABLE EXISTENCE
        # ----------------------------------------------------

        if not spark.catalog.tableExists(full_table_name):

            result["audit_status"] = "MISSING_TABLE"

            result["audit_details"] = (
                f"Target table does not exist: {full_table_name}"
            )

            print("Table Status: MISSING")

            return result

        result["table_exists"] = True

        print("Table Status: EXISTS")


        # ----------------------------------------------------
        # B. COUNT TARGET RECORDS
        # ----------------------------------------------------

        target_count = (
            spark.table(full_table_name).count()
        )

        result["total_target_records"] = int(target_count)

        print(f"Target Records: {target_count}")


        # ----------------------------------------------------
        # C. READ DELTA HISTORY
        # ----------------------------------------------------

        history_df = spark.sql(
            f"DESCRIBE HISTORY {full_table_name}"
        )

        # ----------------------------------------------------
        # D. FIND LATEST MERGE OPERATION
        # ----------------------------------------------------

        merge_history = (
            history_df
            .filter(F.upper(F.col("operation")) == "MERGE")
            .orderBy(F.col("timestamp").desc())
        )

        latest_merge = merge_history.first()

        if latest_merge is None:

            result["audit_status"] = "NO_MERGE_HISTORY"

            result["audit_details"] = (
                "No Delta MERGE operation found in table history."
            )

            print("Merge History: NOT FOUND")

            return result

        result["merge_found"] = True

        result["merge_timestamp"] = latest_merge["timestamp"]

        result["merge_operation"] = latest_merge["operation"]

        print("Merge History: FOUND")
        print(
            f"Latest Merge Timestamp: "
            f"{result['merge_timestamp']}"
        )


        # ----------------------------------------------------
        # E. EXTRACT MERGE METRICS
        # ----------------------------------------------------

        metrics = latest_merge["operationMetrics"]

        if metrics is None:
            metrics = {}

        inserted_rows = get_metric(
            metrics,
            "numTargetRowsInserted"
        )

        updated_rows = get_metric(
            metrics,
            "numTargetRowsUpdated"
        )

        deleted_rows = get_metric(
            metrics,
            "numTargetRowsDeleted"
        )

        output_rows = get_metric(
            metrics,
            "numOutputRows"
        )

        result["inserted_rows"] = inserted_rows
        result["updated_rows"] = updated_rows
        result["deleted_rows"] = deleted_rows
        result["output_rows"] = output_rows


        # ----------------------------------------------------
        # F. DISPLAY MERGE METRICS
        # ----------------------------------------------------

        print(f"Inserted Rows: {inserted_rows}")
        print(f"Updated Rows: {updated_rows}")
        print(f"Deleted Rows: {deleted_rows}")
        print(f"Output Rows: {output_rows}")


        # ----------------------------------------------------
        # G. FINAL AUDIT STATUS
        # ----------------------------------------------------

        result["audit_status"] = "SUCCESS"

        result["audit_details"] = (
            "Target table exists and latest Delta MERGE "
            "history was successfully audited."
        )

        print("Audit Status: SUCCESS")

        return result

    except Exception as e:

        result["audit_status"] = "FAILED"

        result["audit_details"] = str(e)[:4000]

        print(f"Audit Status: FAILED")
        print(f"Error: {str(e)}")

        return result


# ============================================================
# 8. EXECUTE AUDIT FOR ALL DATASETS
# ============================================================

audit_results = []

for dataset_name, table_name in DATASETS.items():

    audit_result = audit_delta_merge(
        dataset_name,
        table_name
    )

    audit_results.append(audit_result)


# ============================================================
# 9. CREATE AUDIT DATAFRAME
# ============================================================

audit_df = spark.createDataFrame(
    audit_results,
    schema=audit_schema
)

print("\n" + "=" * 70)
print("AUDIT RESULTS")
print("=" * 70)

display(
    audit_df.orderBy("dataset_name")
)



# ============================================================
# 10. SAVE AUDIT REPORT - SCHEMA SAFE
# ============================================================

if spark.catalog.tableExists(AUDIT_TABLE):

    print("Existing audit table found.")

    (
        audit_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(AUDIT_TABLE)
    )

    print("Audit report appended with schema migration.")

else:

    (
        audit_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(AUDIT_TABLE)
    )

    print("New audit report table created.")

print(f"Audit Report Saved: {AUDIT_TABLE}")



# ============================================================
# 11. EXECUTION SUMMARY
# ============================================================

total_datasets = len(audit_results)

successful_audits = sum(
    1 for row in audit_results
    if row["audit_status"] == "SUCCESS"
)

failed_audits = sum(
    1 for row in audit_results
    if row["audit_status"] == "FAILED"
)

missing_tables = sum(
    1 for row in audit_results
    if row["audit_status"] == "MISSING_TABLE"
)

no_merge_history = sum(
    1 for row in audit_results
    if row["audit_status"] == "NO_MERGE_HISTORY"
)


# ============================================================
# 12. FINAL EXECUTION REPORT
# ============================================================

print("\n")
print("=" * 70)
print("FINAL EXECUTION SUMMARY")
print("=" * 70)

print(f"Total Datasets: {total_datasets}")

print(f"Successful Audits: {successful_audits}")

print(f"Failed Audits: {failed_audits}")

print(f"Missing Tables: {missing_tables}")

print(f"No Merge History Found: {no_merge_history}")

print(f"Audit Table: {AUDIT_TABLE}")

print("=" * 70)


# ============================================================
# 13. FINAL EXECUTION STATUS
# ============================================================

if successful_audits == total_datasets:

    print("EXECUTION STATUS: COMPLETED")

elif successful_audits + missing_tables + no_merge_history == total_datasets:

    print("EXECUTION STATUS: COMPLETED WITH WARNINGS")

else:

    print("EXECUTION STATUS: COMPLETED WITH ERRORS")

print("=" * 70)


# ============================================================
# 14. DISPLAY SAVED AUDIT REPORT
# ============================================================

print("\nLATEST AUDIT REPORT")

display(
    spark.table(AUDIT_TABLE)
    .filter(F.col("run_id") == RUN_ID)
    .orderBy("dataset_name")
)



### After running the notebook:


**Run this SQL query in a SQL cell to inspect the audit report:**

In [0]:
%sql
SELECT
    dataset_name,
    silver_table,
    delta_version,
    operation,
    operation_timestamp,
    inserted_rows,
    updated_rows,
    deleted_rows,
    audit_status,
    audit_message,
    audit_timestamp
FROM workspace.ecommerce_silver.incremental_delta_merge_audit_report
ORDER BY audit_timestamp DESC, dataset_name;


### Merge execution report

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_silver.delta_merge_execution_report
ORDER BY execution_timestamp DESC;


### Insert and update totals

In [0]:
%sql

SELECT
    dataset_name,
    SUM(inserted_rows) AS total_inserted,
    SUM(updated_rows) AS total_updated,
    SUM(deleted_rows) AS total_deleted
FROM workspace.ecommerce_silver.delta_merge_execution_report
WHERE execution_status = 'SUCCESS'
GROUP BY dataset_name
ORDER BY dataset_name;


### C. Check inventory composite-key duplicates

In [0]:
%sql

SELECT
    product_id,
    warehouse_id,
    snapshot_date,
    COUNT(*) AS duplicate_count
FROM workspace.ecommerce_silver.silver_inventory
GROUP BY
    product_id,
    warehouse_id,
    snapshot_date
HAVING COUNT(*) > 1;